## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers.core import Dense,Dropout,Activation,Flatten
from keras.layers.convolutional import Conv2D,MaxPooling2D
from keras.utils import np_utils
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

Using TensorFlow backend.


In [2]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default
#tf.enable_eager_execution()

### Collect Data

In [3]:
import pandas as pd

In [4]:
df= pd.read_csv('prices.csv')

### Check all columns in the dataset

In [5]:
df.shape

(851264, 7)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [7]:
df = df.drop(['date','symbol'],axis=1)

In [8]:
df.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [9]:
df = df.iloc[0:1000, :]

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
open      1000 non-null float64
close     1000 non-null float64
low       1000 non-null float64
high      1000 non-null float64
volume    1000 non-null float64
dtypes: float64(5)
memory usage: 39.1 KB


In [11]:
df['volume'] = df['volume']/1000000

In [12]:
df.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X = df.drop("volume", axis=1)
y = df["volume"]
test_size = 0.30 # taking 70:30 training and test set
seed = 7  # Random numbmer seeding for reapeatability of the code
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)

In [15]:
y_test.sample(n=300)

186     1.5045
113     0.4048
935     6.0706
259     2.1027
709     2.8247
172     0.6411
750     2.5758
50      0.8093
907     6.0319
43      0.8445
846     3.4566
296     1.3012
284     7.5999
779     7.1893
810     0.5110
396     3.7810
693     8.1710
972     6.2051
473     0.2860
168     0.4873
57      0.5910
718     4.1860
148     1.8652
801     2.3527
511     3.8114
780     0.1988
326     6.7109
142     0.2755
98      0.4621
222     0.8190
        ...   
466     0.7948
171     0.7283
334     0.8229
794     2.1160
351     0.4812
355     7.9060
790     5.6972
362    14.6768
988     0.3699
675     3.8972
768     8.8678
696    39.3357
482     2.7281
799     1.9360
736     0.5834
63      2.0353
802     0.0100
712    13.4727
122     1.0497
590     1.1697
782    16.9736
431     1.9084
602     1.2330
297     2.1761
747     0.3246
134     0.4622
125     0.7614
839     1.5352
127     0.5911
621     0.8668
Name: volume, Length: 300, dtype: float64

#### Convert Training and Test Data to numpy float32 arrays


In [16]:
X_train = np.array(X_train).astype(np.float32)

In [17]:
X_test = np.array(X_test).astype(np.float32)

In [18]:
y_train = np.array(y_train).astype(np.float32)

In [19]:
y_test = np.array(y_test).astype(np.float32)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [20]:
from sklearn.preprocessing import Normalizer

In [21]:
X_train = Normalizer().fit_transform(X_train)

In [22]:
X_train

array([[0.50021225, 0.5003598 , 0.49549326, 0.503899  ],
       [0.5023256 , 0.50103617, 0.4931476 , 0.5034255 ],
       [0.5004301 , 0.49899533, 0.4962851 , 0.5042563 ],
       ...,
       [0.5001929 , 0.49977148, 0.4976645 , 0.50236005],
       [0.50253594, 0.49852157, 0.496339  , 0.50257486],
       [0.50121456, 0.4987676 , 0.49595362, 0.5040285 ]], dtype=float32)

In [23]:
X_test=Normalizer().fit_transform(X_test)

In [24]:
X_test

array([[0.5063693 , 0.49396917, 0.4904527 , 0.50896037],
       [0.49942613, 0.50043815, 0.49706477, 0.5030525 ],
       [0.4963037 , 0.50399053, 0.49479976, 0.50482607],
       ...,
       [0.49629417, 0.50301725, 0.49192008, 0.5086063 ],
       [0.49930164, 0.5008185 , 0.49753195, 0.50233537],
       [0.49550715, 0.50393987, 0.49483255, 0.50562644]], dtype=float32)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [25]:
tf.reset_default_graph()

In [26]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1)

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

In [27]:
#W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

Instructions for updating:
Colocations handled automatically by placer.


2.Define a function to calculate prediction

In [28]:
y = tf.add(tf.matmul(x_n,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [29]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

In [30]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [31]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [32]:
for epoch in range(training_epochs):
        #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train, y_:y_train})
    
    if epoch % 1 == 0:
        print ('Training loss at every iteration: ', epoch, ' is ', train_loss)

Training loss at every iteration:  0  is  305.42902
Training loss at every iteration:  1  is  298.32843
Training loss at every iteration:  2  is  292.82974
Training loss at every iteration:  3  is  288.57153
Training loss at every iteration:  4  is  285.2738
Training loss at every iteration:  5  is  282.72012
Training loss at every iteration:  6  is  280.74277
Training loss at every iteration:  7  is  279.21103
Training loss at every iteration:  8  is  278.02518
Training loss at every iteration:  9  is  277.10678
Training loss at every iteration:  10  is  276.39557
Training loss at every iteration:  11  is  275.8448
Training loss at every iteration:  12  is  275.41818
Training loss at every iteration:  13  is  275.08798
Training loss at every iteration:  14  is  274.8322
Training loss at every iteration:  15  is  274.634
Training loss at every iteration:  16  is  274.4808
Training loss at every iteration:  17  is  274.3619
Training loss at every iteration:  18  is  274.2699
Training lo

In [33]:
sess.close()

In [34]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

In [35]:
for epoch in range(training_epochs):
        #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train, y_:y_train})
    
    if epoch % 5 == 0:
        print ('Training loss at every iteration: ', epoch, ' is ', train_loss)
        #numpy_array = W.eval(session=sess)

Training loss at every iteration:  0  is  305.42902
Training loss at every iteration:  5  is  282.72012
Training loss at every iteration:  10  is  276.39557
Training loss at every iteration:  15  is  274.634
Training loss at every iteration:  20  is  274.14355
Training loss at every iteration:  25  is  274.0067
Training loss at every iteration:  30  is  273.96872
Training loss at every iteration:  35  is  273.95813
Training loss at every iteration:  40  is  273.95538
Training loss at every iteration:  45  is  273.95447
Training loss at every iteration:  50  is  273.95425
Training loss at every iteration:  55  is  273.95407
Training loss at every iteration:  60  is  273.9541
Training loss at every iteration:  65  is  273.95413
Training loss at every iteration:  70  is  273.95416
Training loss at every iteration:  75  is  273.9542
Training loss at every iteration:  80  is  273.95425
Training loss at every iteration:  85  is  273.95428
Training loss at every iteration:  90  is  273.95425


In [36]:
Weight_array = W.eval(session=sess)
bias_array = b.eval(session=sess)

In [37]:
sess.close()

### Get the shapes and values of W and b

In [38]:
print('Shape of W =',W)
print('Shape of b =',b)
print('Weights Values =',Weight_array)
print('Bias Value =',bias_array)

Shape of W = <tf.Variable 'Weights:0' shape=(4, 1) dtype=float32_ref>
Shape of b = <tf.Variable 'Bias:0' shape=(1,) dtype=float32_ref>
Weights Values = [[1.4007461]
 [1.4055865]
 [1.3863369]
 [1.4173869]]
Bias Value = [2.8051822]


### Model Prediction on 1st Examples in Test Dataset

In [39]:
ypred = tf.add(tf.matmul(X_test,Weight_array),bias_array,name='output')

In [40]:
#Lets start graph Execution
sess1 = tf.Session()
ypred.eval(session=sess1)

array([[5.61012  ],
       [5.61028  ],
       [5.6102734],
       [5.6102104],
       [5.610093 ],
       [5.610279 ],
       [5.610261 ],
       [5.6102896],
       [5.6102657],
       [5.610283 ],
       [5.6102867],
       [5.6102858],
       [5.6102962],
       [5.6102734],
       [5.610194 ],
       [5.6102886],
       [5.610196 ],
       [5.610256 ],
       [5.609956 ],
       [5.6102614],
       [5.610283 ],
       [5.6102786],
       [5.610194 ],
       [5.61026  ],
       [5.610219 ],
       [5.61022  ],
       [5.6099815],
       [5.6102676],
       [5.6098657],
       [5.609933 ],
       [5.610265 ],
       [5.610116 ],
       [5.6102824],
       [5.6102037],
       [5.6102133],
       [5.6102877],
       [5.610281 ],
       [5.6102667],
       [5.610269 ],
       [5.6102657],
       [5.610289 ],
       [5.61026  ],
       [5.6102734],
       [5.610262 ],
       [5.610238 ],
       [5.610199 ],
       [5.6102457],
       [5.6102514],
       [5.610199 ],
       [5.610281 ],


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [41]:
iris= pd.read_csv('Iris.csv')

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [42]:
iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [44]:
X = iris.iloc[:,1:5].values
y = iris.iloc[:,5].values

from sklearn.preprocessing import LabelEncoder
encoder =  LabelEncoder()
y1 = encoder.fit_transform(y)

Y = pd.get_dummies(y1).values

### Splitting the data into feature set and target set

In [45]:
from sklearn.model_selection import train_test_split
X_train,X_test, y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [47]:
model = Sequential()
model.add(Dense(12, input_dim=4, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

### Model Training 

In [48]:
model.fit(X_train, y_train, epochs=100)

Instructions for updating:
Use tf.cast instead.
Epoch 1/100
120/120 [==============================] - 0s 2ms/step - loss: 2.1491 - acc: 0.3083
Epoch 2/100
120/120 [==============================] - 0s 58us/step - loss: 1.1245 - acc: 0.3667
Epoch 3/100
120/120 [==============================] - 0s 50us/step - loss: 0.9297 - acc: 0.5750
Epoch 4/100
120/120 [==============================] - 0s 84us/step - loss: 0.9050 - acc: 0.4083
Epoch 5/100
120/120 [==============================] - 0s 50us/step - loss: 0.8834 - acc: 0.3917
Epoch 6/100
120/120 [==============================] - 0s 42us/step - loss: 0.8767 - acc: 0.3833
Epoch 7/100
120/120 [==============================] - 0s 54us/step - loss: 0.8678 - acc: 0.3833
Epoch 8/100
120/120 [==============================] - 0s 72us/step - loss: 0.8476 - acc: 0.4000
Epoch 9/100
120/120 [==============================] - 0s 50us/step - loss: 0.8362 - acc: 0.3917
Epoch 10/100
120/120 [==============================] - 0s 50us/step - loss: 0.8

120/120 [==============================] - 0s 67us/step - loss: 0.3949 - acc: 0.9250
Epoch 82/100
120/120 [==============================] - 0s 50us/step - loss: 0.3929 - acc: 0.9000
Epoch 83/100
120/120 [==============================] - 0s 50us/step - loss: 0.3929 - acc: 0.9167
Epoch 84/100
120/120 [==============================] - 0s 50us/step - loss: 0.3890 - acc: 0.8917
Epoch 85/100
120/120 [==============================] - 0s 67us/step - loss: 0.3873 - acc: 0.9500
Epoch 86/100
120/120 [==============================] - 0s 50us/step - loss: 0.3849 - acc: 0.9167
Epoch 87/100
120/120 [==============================] - 0s 42us/step - loss: 0.3855 - acc: 0.9167
Epoch 88/100
120/120 [==============================] - 0s 58us/step - loss: 0.3800 - acc: 0.9417
Epoch 89/100
120/120 [==============================] - 0s 58us/step - loss: 0.3814 - acc: 0.9167
Epoch 90/100
120/120 [==============================] - 0s 67us/step - loss: 0.3768 - acc: 0.8917
Epoch 91/100
120/120 [===========

### Model Prediction

In [49]:
y_predict = model.predict(X)
y_predict

array([[9.21900809e-01, 7.65752494e-02, 1.52386376e-03],
       [9.00922298e-01, 9.63693857e-02, 2.70838127e-03],
       [9.06772852e-01, 9.07676071e-02, 2.45958520e-03],
       [8.77057731e-01, 1.18790761e-01, 4.15149564e-03],
       [9.21721816e-01, 7.67310932e-02, 1.54712377e-03],
       [9.20262814e-01, 7.84148052e-02, 1.32239715e-03],
       [9.02094007e-01, 9.53031704e-02, 2.60282541e-03],
       [9.12793994e-01, 8.52967054e-02, 1.90925074e-03],
       [8.68605971e-01, 1.26252130e-01, 5.14184078e-03],
       [8.98336351e-01, 9.89218429e-02, 2.74177315e-03],
       [9.31250215e-01, 6.76691905e-02, 1.08065293e-03],
       [8.91271293e-01, 1.05777435e-01, 2.95129488e-03],
       [9.01874304e-01, 9.54244807e-02, 2.70122173e-03],
       [8.90978754e-01, 1.05304249e-01, 3.71689792e-03],
       [9.42410946e-01, 5.69427647e-02, 6.46271859e-04],
       [9.46843028e-01, 5.26051745e-02, 5.51777892e-04],
       [9.34973955e-01, 6.40953779e-02, 9.30647715e-04],
       [9.22214806e-01, 7.62698

### Save the Model

In [50]:
import pickle
# Save the trained model as a pickle file. 
pickle.dump(model, open("model.pkl","wb"))